In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [1]:
!pip -q install -U transformers datasets evaluate accelerate scikit-learn scipy pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 101.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 10.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requir

In [2]:
%%writefile single_task_distilbert_stsb.py
import os
import json
import time
import shutil
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from scipy.stats import pearsonr, spearmanr


# =========================
# Config
# =========================
MODEL_NAME = "distilbert-base-uncased"
TASK_NAME = "stsb"   # supported: sst2, qqp, stsb
SETTING = "single_task"

BATCH_SIZE = 32
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
MAX_LENGTH = 256
NUM_WORKERS = 2
EARLY_STOPPING_PATIENCE = 3
KEEP_LAST_K_CHECKPOINTS = 2
SEED = 42

BASE_DIR = Path("/content/drive/MyDrive/single_task_runs")
RUN_NAME = f"{SETTING}_{MODEL_NAME.replace('/', '_')}_{TASK_NAME}"
RUN_DIR = BASE_DIR / RUN_NAME

CHECKPOINT_DIR = RUN_DIR / "checkpoints"
BEST_MODEL_DIR = RUN_DIR / "best_model"
FINAL_MODEL_DIR = RUN_DIR / "final_model"
LOG_CSV_PATH = RUN_DIR / "history.csv"
LOG_JSON_PATH = RUN_DIR / "history.json"
STATE_PATH = RUN_DIR / "training_state.json"


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dirs():
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    BEST_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)


def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return trainable_params, total_params


def get_task_info(task_name):
    task_name = task_name.lower()
    if task_name == "sst2":
        return {
            "task_type": "classification",
            "dataset_loader": ("glue", "sst2"),
            "text_cols": ("sentence", None),
            "label_col": "label",
            "num_labels": 2,
            "primary_metric": "accuracy",
        }
    elif task_name == "qqp":
        return {
            "task_type": "classification",
            "dataset_loader": ("glue", "qqp"),
            "text_cols": ("question1", "question2"),
            "label_col": "label",
            "num_labels": 2,
            "primary_metric": "accuracy",
        }
    elif task_name == "stsb":
        return {
            "task_type": "regression",
            "dataset_loader": ("glue", "stsb"),
            "text_cols": ("sentence1", "sentence2"),
            "label_col": "label",
            "num_labels": 1,
            "primary_metric": "pearson",
        }
    else:
        raise ValueError(f"Unsupported task: {task_name}. Supported: sst2, qqp, stsb")


def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def save_history(history):
    df = pd.DataFrame(history)
    df.to_csv(LOG_CSV_PATH, index=False)
    save_json(history, LOG_JSON_PATH)


def get_latest_checkpoint():
    if not CHECKPOINT_DIR.exists():
        return None
    ckpts = [p for p in CHECKPOINT_DIR.iterdir() if p.is_dir() and p.name.startswith("epoch_")]
    if not ckpts:
        return None
    ckpts = sorted(ckpts, key=lambda x: int(x.name.split("_")[-1]))
    return ckpts[-1]


def cleanup_old_checkpoints(keep_k=2):
    ckpts = [p for p in CHECKPOINT_DIR.iterdir() if p.is_dir() and p.name.startswith("epoch_")]
    ckpts = sorted(ckpts, key=lambda x: int(x.name.split("_")[-1]))
    while len(ckpts) > keep_k:
        oldest = ckpts.pop(0)
        shutil.rmtree(oldest, ignore_errors=True)


def save_checkpoint(epoch, model, tokenizer, optimizer, scheduler, history, best_metric_so_far, patience_counter):
    ckpt_dir = CHECKPOINT_DIR / f"epoch_{epoch}"
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    model.save_pretrained(ckpt_dir)
    tokenizer.save_pretrained(ckpt_dir)

    torch.save(
        {
            "epoch": epoch,
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_metric_so_far": best_metric_so_far,
            "patience_counter": patience_counter,
            "history": history,
            "model_name": MODEL_NAME,
            "task_name": TASK_NAME,
            "setting": SETTING,
        },
        ckpt_dir / "trainer_state.pt",
    )

    save_json(
        {
            "epoch": epoch,
            "best_metric_so_far": best_metric_so_far,
            "patience_counter": patience_counter,
            "history_length": len(history),
            "model_name": MODEL_NAME,
            "task_name": TASK_NAME,
            "setting": SETTING,
        },
        STATE_PATH,
    )

    cleanup_old_checkpoints(KEEP_LAST_K_CHECKPOINTS)


def get_na():
    return None


def load_and_prepare_data(tokenizer, task_info):
    dataset_name, subset_name = task_info["dataset_loader"]
    raw = load_dataset(dataset_name, subset_name)

    text_a, text_b = task_info["text_cols"]
    label_col = task_info["label_col"]
    task_type = task_info["task_type"]

    def preprocess_fn(examples):
        if text_b is None:
            enc = tokenizer(examples[text_a], truncation=True, max_length=MAX_LENGTH)
        else:
            enc = tokenizer(examples[text_a], examples[text_b], truncation=True, max_length=MAX_LENGTH)

        labels = examples[label_col]
        if task_type == "classification":
            enc["labels"] = labels
        else:
            enc["labels"] = [float(x) for x in labels]
        return enc

    encoded = raw.map(preprocess_fn, batched=True, remove_columns=raw["train"].column_names)

    if "validation" in encoded:
        val_split = "validation"
    elif "validation_matched" in encoded:
        val_split = "validation_matched"
    else:
        raise ValueError("No validation split found.")

    return encoded["train"], encoded[val_split]


@torch.no_grad()
def evaluate_classification(model, dataloader, device):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        logits = outputs.logits

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=-1)

        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_labels.extend(batch["labels"].detach().cpu().numpy().tolist())

    eval_loss = total_loss / max(len(dataloader), 1)
    accuracy = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    precision = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    recall = recall_score(all_labels, all_preds, average="macro", zero_division=0)

    return {
        "eval_loss": eval_loss,
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "precision": precision,
        "recall": recall,
        "pearson": get_na(),
        "spearman": get_na(),
        "best_metric_candidate": accuracy,
    }


@torch.no_grad()
def evaluate_regression(model, dataloader, device):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        logits = outputs.logits.squeeze(-1)

        total_loss += loss.item()
        all_preds.extend(logits.detach().cpu().numpy().tolist())
        all_labels.extend(batch["labels"].detach().cpu().numpy().tolist())

    eval_loss = total_loss / max(len(dataloader), 1)

    if len(set(all_preds)) <= 1 or len(set(all_labels)) <= 1:
        pearson = 0.0
        spearman = 0.0
    else:
        pearson = float(pearsonr(all_labels, all_preds)[0])
        spearman = float(spearmanr(all_labels, all_preds)[0])

    return {
        "eval_loss": eval_loss,
        "accuracy": get_na(),
        "macro_f1": get_na(),
        "precision": get_na(),
        "recall": get_na(),
        "pearson": pearson,
        "spearman": spearman,
        "best_metric_candidate": pearson,
    }


def main():
    set_seed(SEED)
    ensure_dirs()

    task_info = get_task_info(TASK_NAME)
    task_type = task_info["task_type"]

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("=" * 100)
    print(f"MODEL_NAME: {MODEL_NAME}")
    print(f"TASK_NAME : {TASK_NAME}")
    print(f"TASK_TYPE : {task_type}")
    print(f"DEVICE    : {device}")
    print(f"RUN_DIR   : {RUN_DIR}")
    print("Training will continue until early stopping patience=3 is met.")
    print("No fixed max_epochs is used.")
    print("=" * 100)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

    train_ds, val_ds = load_and_prepare_data(tokenizer, task_info)

    collator = DataCollatorWithPadding(tokenizer=tokenizer)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collator,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collator,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=task_info["num_labels"],
        problem_type="regression" if task_type == "regression" else None,
    )
    model.to(device)

    trainable_params, total_params = count_parameters(model)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    steps_per_epoch = len(train_loader)
    warmup_steps = max(1, int(steps_per_epoch * WARMUP_RATIO))
    scheduler = get_linear_schedule_with_warmup(
        optimizer=optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=10**12,
    )

    history = []
    start_epoch = 1
    best_metric_so_far = -float("inf")
    patience_counter = 0

    latest_ckpt = get_latest_checkpoint()
    if latest_ckpt is not None:
        print(f"Resuming from checkpoint: {latest_ckpt}")
        model = AutoModelForSequenceClassification.from_pretrained(latest_ckpt)
        model.to(device)

        trainer_state_path = latest_ckpt / "trainer_state.pt"
        if trainer_state_path.exists():
            state = torch.load(trainer_state_path, map_location=device)
            optimizer.load_state_dict(state["optimizer_state_dict"])
            scheduler.load_state_dict(state["scheduler_state_dict"])
            history = state.get("history", [])
            best_metric_so_far = state.get("best_metric_so_far", -float("inf"))
            patience_counter = state.get("patience_counter", 0)
            start_epoch = state.get("epoch", 0) + 1

    print(f"Train samples: {len(train_ds)} | Validation samples: {len(val_ds)}")
    print(f"Trainable params: {trainable_params:,} | Total params: {total_params:,}")

    epoch = start_epoch
    while True:
        model.train()
        epoch_start_time = time.time()
        running_loss = 0.0

        print("\n" + "=" * 100)
        print(f"Epoch {epoch} started")
        print("=" * 100)

        for step, batch in enumerate(train_loader, start=1):
            batch = {k: v.to(device) for k, v in batch.items()}

            optimizer.zero_grad()
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()

            optimizer.step()
            scheduler.step()

            running_loss += loss.item()

            if step % 500 == 0 or step == len(train_loader):
                print(f"Epoch {epoch} | Step {step}/{len(train_loader)} | Train Loss: {running_loss / step:.6f}")

        train_loss = running_loss / max(len(train_loader), 1)
        time_per_epoch = time.time() - epoch_start_time

        if task_type == "classification":
            eval_metrics = evaluate_classification(model, val_loader, device)
        else:
            eval_metrics = evaluate_regression(model, val_loader, device)

        current_metric = eval_metrics["best_metric_candidate"]
        is_new_best = current_metric > best_metric_so_far

        if is_new_best:
            best_metric_so_far = current_metric
            patience_counter = 0

            if BEST_MODEL_DIR.exists():
                shutil.rmtree(BEST_MODEL_DIR)
            model.save_pretrained(BEST_MODEL_DIR)
            tokenizer.save_pretrained(BEST_MODEL_DIR)
        else:
            patience_counter += 1

        row = {
            "epoch": epoch,
            "train_loss": float(train_loss),
            "eval_loss": float(eval_metrics["eval_loss"]),
            "accuracy": eval_metrics["accuracy"],
            "macro_f1": eval_metrics["macro_f1"],
            "precision": eval_metrics["precision"],
            "recall": eval_metrics["recall"],
            "pearson": eval_metrics["pearson"],
            "spearman": eval_metrics["spearman"],
            "time_per_epoch": float(time_per_epoch),
            "trainable_params": int(trainable_params),
            "total_params": int(total_params),
            "model_name": MODEL_NAME,
            "dataset_name": TASK_NAME,
            "setting": SETTING,
            "best_metric_so_far": float(best_metric_so_far),
            "patience_counter": int(patience_counter),
            "is_new_best": bool(is_new_best),
        }
        history.append(row)
        save_history(history)

        save_checkpoint(
            epoch=epoch,
            model=model,
            tokenizer=tokenizer,
            optimizer=optimizer,
            scheduler=scheduler,
            history=history,
            best_metric_so_far=best_metric_so_far,
            patience_counter=patience_counter,
        )

        print("\nValidation results:")
        print(json.dumps(row, indent=2, ensure_ascii=False, default=str))

        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print("\nEarly stopping triggered.")
            print(f"Validation metric did not improve for {EARLY_STOPPING_PATIENCE} consecutive epochs.")
            break

        epoch += 1

    print("\nSaving final model...")
    if FINAL_MODEL_DIR.exists():
        shutil.rmtree(FINAL_MODEL_DIR)
    model.save_pretrained(FINAL_MODEL_DIR)
    tokenizer.save_pretrained(FINAL_MODEL_DIR)

    save_history(history)

    print("\nTraining completed.")
    print(f"History CSV : {LOG_CSV_PATH}")
    print(f"History JSON: {LOG_JSON_PATH}")
    print(f"Best model  : {BEST_MODEL_DIR}")
    print(f"Final model : {FINAL_MODEL_DIR}")
    print(f"Checkpoints : {CHECKPOINT_DIR}")


if __name__ == "__main__":
    main()

Writing single_task_distilbert_stsb.py


In [3]:
!python /content/single_task_distilbert_stsb.py

MODEL_NAME: distilbert-base-uncased
TASK_NAME : stsb
TASK_TYPE : regression
DEVICE    : cuda
RUN_DIR   : /content/drive/MyDrive/single_task_runs/single_task_distilbert-base-uncased_stsb
Training will continue until early stopping patience=3 is met.
No fixed max_epochs is used.
config.json: 100% 483/483 [00:00<00:00, 1.22MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 151kB/s]
vocab.txt: 232kB [00:00, 3.14MB/s]
tokenizer.json: 466kB [00:00, 10.4MB/s]
README.md: 35.3kB [00:00, 56.3MB/s]
stsb/train-00000-of-00001.parquet: 100% 502k/502k [00:00<00:00, 714kB/s] 
stsb/validation-00000-of-00001.parquet: 100% 151k/151k [00:00<00:00, 367kB/s]
stsb/test-00000-of-00001.parquet: 100% 114k/114k [00:00<00:00, 279kB/s]
Generating train split: 100% 5749/5749 [00:00<00:00, 567205.82 examples/s]
Generating validation split: 100% 1500/1500 [00:00<00:00, 510835.99 examples/s]
Generating test split: 100% 1379/1379 [00:00<00:00, 545192.31 examples/s]
Map: 100% 5749/5749 [00:00<00:00, 5759.38 examp